# Dataset 检查 / Dataset Inspection

目的：搞清楚 `Dataset/` 下到底有什么、列长啥样、SFT 输出末尾是不是字面 `eos`、长度分布怎样。

重点核查项：
1. **文件目录结构** —— Processed / Processed_plain / Unprocessed 各是什么
2. **列定义** —— `prompt` / `output` / `split` / `Task Description` / `input` 是什么
3. **EOS 字面字符串** —— output 末尾是不是真的有 `eos`（这是 RL 训练时模型啰嗦的元凶猜想）
4. **长度分布** —— prompt / output 多长，是否会顶到 `max_prompt_length` / `max_new_tokens`
5. **split 分布** —— train / validation / test 各有多少
6. **去重** —— prompt 是否有重复（数据泄漏检查）

In [ ]:
from pathlib import Path
import pandas as pd

DATA_ROOT = Path('.').resolve()
print('Working dir:', DATA_ROOT)

for sub in ['Processed', 'Processed_plain', 'Unprocessed']:
    p = DATA_ROOT / sub
    if p.exists():
        print(f'\n[{sub}]')
        for f in sorted(p.iterdir()):
            if f.is_file():
                size_mb = f.stat().st_size / 1e6
                print(f'  {f.name:<48} {size_mb:>8.2f} MB')

## 1. 主训练文件：`Processed/train_33_69_84_nodes.csv`

RL 训练用的就是这个（见 `RL/train_grpo_adapter_*.sh` 的 `--data_path`）。

In [ ]:
MAIN_CSV = DATA_ROOT / 'Processed' / 'train_33_69_84_nodes.csv'
df = pd.read_csv(MAIN_CSV)
print(f'Total rows: {len(df):,}')
print(f'Columns:    {list(df.columns)}')
df.head(2)

In [ ]:
# split 分布
if 'split' in df.columns:
    print(df['split'].value_counts())
else:
    print('(no split column)')

## 2. 列定义：每一列长啥样

我们看一行的全文，理解 5 列各自是什么。

In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', None)

row = df.iloc[0]
for col in df.columns:
    val = str(row[col])
    print(f'\n{"="*100}\n{col}  ({len(val)} chars)\n{"="*100}')
    print(val)

## 3. 🔍 关键检查：output 末尾是不是字面 `eos`

这是 RL 训练时模型胡乱写 `eos\nLists all predicted...` 这类废话的元凶猜想。

如果 SFT 数据末尾是字面 `eos`，模型就会学到「先写 System Loss，再写 e、o、s 三个字母」当成普通 token，**根本不触发真 EOS**。

In [ ]:
# 抽 5 条样本看 output 完整尾部（不截断）
print('— 抽 5 条样本看 output 尾部（最后 300 字符）—')
for i in df.sample(5, random_state=0).index:
    out = str(df.loc[i, 'output'])
    tail = out[-300:] if len(out) > 300 else out
    print(f'\n--- row {i} (total {len(out)} chars) ---')
    print(repr(tail))

In [ ]:
# 统计 output 末尾模式
import re

def classify_tail(text: str) -> str:
    s = text.rstrip()
    if s.endswith('eos'):
        return 'literal_eos'
    if s.endswith('<|eos|>') or s.endswith('<|endoftext|>') or s.endswith('<|eot_id|>'):
        return 'special_token_str'
    if re.search(r'System Loss=[\d.]+\s*$', s):
        return 'clean_system_loss'
    return 'other'

tail_kind = df['output'].astype(str).map(classify_tail)
print(tail_kind.value_counts())
print(f'\nFraction literal "eos": {(tail_kind == "literal_eos").mean():.4f}')

In [ ]:
# 看几条完整 output（不截断）
for i in df.head(3).index:
    out = df.loc[i, 'output']
    print(f'\n{"="*100}\nrow {i}  (output length = {len(out)} chars)\n{"="*100}')
    print(out)

## 4. 长度分布：prompt / output 字符数与估计 token 数

RL 训练用 `--max_prompt_length 3072 --max_new_tokens 512`。要确认数据没顶到上限。

In [ ]:
df_len = pd.DataFrame({
    'prompt_chars': df['prompt'].astype(str).str.len(),
    'output_chars': df['output'].astype(str).str.len(),
})
df_len.describe(percentiles=[.5, .9, .95, .99]).round(1)

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
df_len['prompt_chars'].hist(bins=60, ax=axes[0])
axes[0].set_title('prompt char length')
axes[0].set_xlabel('chars')
df_len['output_chars'].hist(bins=60, ax=axes[1])
axes[1].set_title('output char length')
axes[1].set_xlabel('chars')
plt.tight_layout()

In [ ]:
# 粗略估算 token 数（≈ chars / 3.5 for English+numbers，仅作 sanity check）
# 真 token 数要用 tokenizer，下一 cell 做
print('Char limits to be aware of:')
print(f'  max_prompt_length=3072 tokens ≈ {3072*3.5:.0f} chars')
print(f'  max_new_tokens   =512  tokens ≈ {512*3.5:.0f} chars')
print()
print('Fraction exceeding rough char budget:')
print(f'  prompt_chars > 10000:  {(df_len.prompt_chars > 10000).mean():.4f}')
print(f'  output_chars > 1500:   {(df_len.output_chars > 1500).mean():.4f}')

### 真 token 长度（可选，需要本地有 tokenizer）

In [ ]:
# 取一个 1000 行子样本算真实 token，避免全量太慢
try:
    from transformers import AutoTokenizer
    TOK_PATH = '../../models/meta-llama/Llama-3.1-8B-Instruct'  # 改成你本地路径
    tok = AutoTokenizer.from_pretrained(TOK_PATH)
    sub = df.sample(1000, random_state=0)
    p_tok = sub['prompt'].astype(str).map(lambda s: len(tok.encode(s)))
    o_tok = sub['output'].astype(str).map(lambda s: len(tok.encode(s)))
    print('prompt token len: ', p_tok.describe(percentiles=[.5, .9, .99]).round(1).to_dict())
    print('output token len: ', o_tok.describe(percentiles=[.5, .9, .99]).round(1).to_dict())
    print(f'\nprompt > 3072 tokens: {(p_tok > 3072).mean():.4f}')
    print(f'output > 512 tokens:  {(o_tok > 512).mean():.4f}')
except Exception as e:
    print('(skipped tokenizer length check —', type(e).__name__, str(e)[:120], ')')

## 5. prompt 重复检查（防数据泄漏）

In [ ]:
n_unique_prompts = df['prompt'].nunique()
print(f'Unique prompts: {n_unique_prompts:,} / {len(df):,}')
print(f'Duplicate fraction: {1 - n_unique_prompts/len(df):.4f}')

# split 之间 prompt 是否重叠
if 'split' in df.columns:
    splits = {s: set(df.loc[df.split == s, 'prompt']) for s in df['split'].unique()}
    keys = list(splits)
    for i, a in enumerate(keys):
        for b in keys[i+1:]:
            overlap = len(splits[a] & splits[b])
            print(f'  {a} ∩ {b}: {overlap}')

## 6. Processed vs Processed_plain：差别是什么

`_plain` 看名字像是没加 `Task Description` 那段说明的版本。对比一下。

In [ ]:
PLAIN_CSV = DATA_ROOT / 'Processed_plain' / 'train_33_69_84_nodes_plain.csv'
df_plain = pd.read_csv(PLAIN_CSV)
print(f'Processed       rows: {len(df):,}, cols: {list(df.columns)}')
print(f'Processed_plain rows: {len(df_plain):,}, cols: {list(df_plain.columns)}')

In [ ]:
# 拿同一行（按 id 对齐）对比 prompt 和 output 完整内容
if 'id' in df.columns and 'id' in df_plain.columns:
    common_id = list(set(df['id']) & set(df_plain['id']))[:1]
    if common_id:
        cid = common_id[0]
        a = df[df.id == cid].iloc[0]
        b = df_plain[df_plain.id == cid].iloc[0]
        for col in ['prompt', 'output']:
            if col in a and col in b:
                print(f'\n{"="*100}\n=== Processed.{col} (id={cid}, {len(str(a[col]))} chars) ===\n{"="*100}')
                print(a[col])
                print(f'\n{"="*100}\n=== Processed_plain.{col} (id={cid}, {len(str(b[col]))} chars) ===\n{"="*100}')
                print(b[col])

## 7. Unprocessed：原始 MATLAB 仿真输出

估计是网格仿真结果的 raw dump，没改成 prompt-output 形式。瞄一眼。

In [ ]:
RAW_CSV = DATA_ROOT / 'Unprocessed' / 'samples_33bus.csv'
if RAW_CSV.exists():
    pd.set_option('display.max_colwidth', None)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)
    df_raw = pd.read_csv(RAW_CSV, nrows=3)
    print('Columns:', list(df_raw.columns))
    with open(RAW_CSV) as f:
        n = sum(1 for _ in f) - 1
    print(f'Total rows: {n:,}\n')
    for i, r in df_raw.iterrows():
        print(f'\n{"="*100}\nrow {i}\n{"="*100}')
        for c in df_raw.columns:
            val = str(r[c])
            print(f'\n[{c}] ({len(val)} chars)')
            print(val)

## 8. 结论汇总（手填）

跑完上面 cell 后，回答这几个问题：

- [ ] output 末尾字面 `eos` 占比是？ ← cell 3 输出
- [ ] prompt p99 token 数？ ← cell 4 输出
- [ ] output p99 token 数？ ← cell 4 输出，超过 512 的比例
- [ ] split 之间是否有 prompt 重叠？ ← cell 5 输出
- [ ] Processed_plain 跟 Processed 主要差啥？ ← cell 6 输出

如果 cell 3 显示 `literal_eos` 占绝大多数 → 实锤了 SFT 数据是模型啰嗦的根源，下一步要么洗数据、要么 vLLM stop strings 兜底。